<a href="https://colab.research.google.com/github/GIRIAYUSH/playing-with-anns/blob/main/notebooks/M2_forward_pass.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

Cloning the Repo

In [ ]:
from google.colab import userdata
import os

TOKEN    = userdata.get('GITHUB_TOEKN')
USERNAME = 'GIRIAYUSH'
REPO     = 'playing-with-anns'
REPO_URL = f'https://{TOKEN}@github.com/{USERNAME}/{REPO}.git'
REPO_PATH = f'/content/{REPO}'

if not os.path.exists(REPO_PATH):
    !git clone {REPO_URL} {REPO_PATH}
else:
    os.chdir(REPO_PATH)
    # Always reset remote to token URL before pulling
    !git remote set-url origin {REPO_URL}
    !git pull origin main

os.chdir(REPO_PATH)
!git remote set-url origin {REPO_URL}   # ensure push also uses token
!git config user.email "giri.ayush2024@gmail.com"
!git config user.name "Ayush Giri"

print(f"Now in: {os.getcwd()}")
!ls

### Forward Pass Implementation

#### Forward Pass
> *"A neural network is just a chain of matrix multiplications with some non-linearities sprinkled in."*
This notebook builds the forward pass from the ground up — no magic, just math and code.

**What I'll cover:**
1. A linear layer from scratch: `output = X @ W.T + b`
2. Stacking layers manually into a deep network
3. Proving why activations matter (spoiler: without them, depth is useless)
4. Clean re-implementation using `nn.Linear` and `nn.Module`
5. Activation functions: ReLU, Sigmoid, Tanh, GELU — implemented & visualised




In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib.pyplot as plt

torch.manual_seed(42) # for reproduceability

print("PyTorch version: ", torch.__version__)

### Part 1 Creating a Linear Layer From Scratch

## Part 1 · Linear Layer from Scratch

A single linear (fully-connected) layer does exactly one thing:

$$\text{output} = X \cdot W^T + b$$

- **X** — input matrix of shape `(batch_size, in_features)`
- **W** — weight matrix of shape `(out_features, in_features)`
- **b** — bias vector of shape `(out_features,)`

We initialise W with small random values and b with zeros.  
This is exactly what `nn.Linear` does under the hood.

In [ ]:
X= torch.randn(4,3) # 4 samples with 3 features

input_dim = 3
output_dim=5

# Weight matrix
W=torch.randn(input_dim,output_dim)*0.1 #scaling to a smaller value
b=torch.zeros(output_dim)

# One forward pass function
output = X  @ W + b #transposed is used to match the output dimensions

print(f"Input shape: {X.shape}")
print(f"Weight shape: {W.shape}")
print("Bias Shape:  {b.shape}")
print("\n Output shape: ", output.shape)
print(f"\n Output: \n", output)

### Stacking Layers Manually

A "deep" network is just multiple linear layers applied one after another.  
The output of one layer becomes the input of the next.

**Network:** `3 → 8 → 8 → 2`  
(input size 3, two hidden layers of size 8, output size 2)

In [ ]:
from binascii import b2a_base64
W1= torch.randn(3,8) * 0.1 ; b1= torch.zeros(8)
W2= torch.randn(8,8) * 0.1 ; b2 = torch.zeros(8)
W3 = torch.randn(8,2) * 0.1 ; b3=torch.zeros(2)

# Input vector: 4 samplese and 3 features

h1 = X @ W1 + b1
h2= h1 @ W2 + b2
out = h2 @ W3 + b3

print("h1 shape :", h1.shape)
print("h2 shape :", h2.shape)
print("out shape:", out.shape)
print("\nFinal output:\n", out)


### Why Activations Matter
## Part 3 · Why Activations Matter

Here's the key insight:

> Without non-linear activations, **stacking N linear layers is mathematically
> identical to a single linear layer.**

Why? Because the product of any number of matrices is still just a matrix.  
Let's prove this empirically below.

In [ ]:
X = torch.randn(4, 3)

# Three separate linear layers (no activations)
W1 = torch.randn(3, 3); b1 = torch.zeros(3)
W2 = torch.randn(3, 3); b2 = torch.zeros(3)
W3 = torch.randn(3, 3); b3 = torch.zeros(3)

# Pass through 3 layers
out_3layers = X @ W1.T @ W2.T @ W3.T   # simplified: ignoring bias for clarity

# Collapsed: W1, W2, W3 multiplied together = one effective matrix
W_collapsed = W3 @ W2 @ W1
out_1layer  = X @ W_collapsed.T

# They should be identical
print("Max difference:", (out_3layers - out_1layer).abs().max().item())
# → near zero: 3 layers without activations = 1 layer

#### Part 4 · Clean Version with `nn.Linear` and `nn.Module`

PyTorch's `nn.Linear` handles the weight init, bias, and matrix multiply for us.  
`nn.Module` is the base class for all neural networks — it tracks parameters
and lets you call `.parameters()`, `.to(device)`, etc.

We'll build the same `3 → 8 → 8 → 2` network, now with ReLU activations
to actually make depth useful.

In [ ]:
class SimpleNet(nn.Module):
    def __init__(self):
        super().__init__()
        self.layer1 = nn.Linear(3, 8)
        self.layer2 = nn.Linear(8, 8)
        self.layer3 = nn.Linear(8, 2)

    def forward(self, x):
        x = F.relu(self.layer1(x))   # linear → activation
        x = F.relu(self.layer2(x))   # linear → activation
        x = self.layer3(x)           # final layer: no activation (raw logits)
        return x

In [ ]:
model = SimpleNet()

X = torch.randn(4, 3)
output = model(X)

print(model)
print("\nInput  shape:", X.shape)
print("Output shape:", output.shape)
print("\nOutput:\n", output)

## Part 5 · Activation Functions

Activations are what give neural networks their power — they introduce
non-linearity so the network can learn complex patterns.

| Activation | Formula | Common Use |
|---|---|---|
| **ReLU** | `max(0, x)` | Default for hidden layers |
| **Sigmoid** | `1 / (1 + e^-x)` | Binary classification output |
| **Tanh** | `(e^x - e^-x) / (e^x + e^-x)` | RNNs, centred around 0 |
| **GELU** | `x · Φ(x)` (smooth ReLU) | Transformers (GPT, BERT) |

Let's implement each from scratch, then compare against PyTorch's built-ins.

In [ ]:
import math

x= torch.linspace(-4,4,200)

# From scratch
def relu_scratch(x):
  return torch.maximum(x,torch.zeros_like(x))

def sigmoid_scratch(x):
  return 1 / (1 + torch.exp(-x))

def tanh_scratch(x):
  return (torch.exp(x) - torch.exp(-x))/ (torch.exp(x) + torch.exp(-x))

def gelu_scratch(x):
  return 0.5 * x * (1 + torch.tanh(math.sqrt(2/math.pi) * (x + 0.044715 * x**3)))

relu_pt    = F.relu(x)
sigmoid_pt = torch.sigmoid(x)
tanh_pt    = torch.tanh(x)
gelu_pt    = F.gelu(x)

print("ReLU    max diff:", (relu_scratch(x)    - relu_pt).abs().max().item())
print("Sigmoid max diff:", (sigmoid_scratch(x) - sigmoid_pt).abs().max().item())
print("Tanh    max diff:", (tanh_scratch(x)    - tanh_pt).abs().max().item())
print("GELU    max diff:", (gelu_scratch(x)    - gelu_pt).abs().max().item())


#### Visualising Activations

Let's see how each activation transforms the same input.  
This makes it intuitive why they behave differently during training.

In [ ]:
x_np = x.numpy()

fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Activation Functions", fontsize=14, fontweight='bold')

configs = [
    ("ReLU",    relu_scratch(x).numpy(),    "steelblue"),
    ("Sigmoid", sigmoid_scratch(x).numpy(), "coral"),
    ("Tanh",    tanh_scratch(x).numpy(),    "seagreen"),
    ("GELU",    gelu_scratch(x).numpy(),    "mediumpurple"),
]

for ax, (name, y, color) in zip(axes, configs):
    ax.plot(x_np, y, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel("x")
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
x_np = x.numpy()
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
fig.suptitle("Activation Functions", fontsize=14, fontweight='bold')

configs = [
    ("ReLU",    F.relu(x).numpy(),    "steelblue"),
    ("Sigmoid", torch.sigmoid(x).numpy(), "coral"),
    ("Tanh",    torch.tanh(x).numpy(),    "seagreen"),
    ("GELU",    F.gelu(x).numpy(),    "mediumpurple"),
]


for ax, (name, y, color) in zip(axes, configs):
    ax.plot(x_np, y, color=color, linewidth=2.5)
    ax.axhline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.axvline(0, color='gray', linewidth=0.7, linestyle='--')
    ax.set_title(name, fontsize=12, fontweight='bold')
    ax.set_xlabel("x")
    ax.set_ylim(-1.5, 1.5)
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()


### Creating a default ANN with 4 layers

In [ ]:
class SimpleANN(nn.Module):
    def __init__(self, input_dim, hidden_dim, output_dim, dropout_rate=0.3):
        super().__init__()

        self.network = nn.Sequential(
            nn.Linear(input_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, hidden_dim),
            nn.ReLU(),
            nn.Dropout(dropout_rate),

            nn.Linear(hidden_dim, output_dim),
        )

    def forward(self, x):
        x = self.network(x)
        return torch.sigmoid(x) #softmax for multiclass classification

#### Performing One forwward pass for a binary classification problem

In [ ]:
X= torch.randn(4,3) #4 samples 3 features
model=SimpleANN(input_dim=3,hidden_dim=4,output_dim=1)
output_forwardpass=model.forward(X)
print(output_forwardpass)